In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 2006
month = 6


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

2006-06-30


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 2006-06-01 12:00:00
end_date 2006-06-02 12:00:00
start_date 2006-06-03 12:00:00
end_date 2006-06-04 12:00:00
start_date 2006-06-05 12:00:00
end_date 2006-06-06 12:00:00
start_date 2006-06-07 12:00:00
end_date 2006-06-08 12:00:00
start_date 2006-06-09 12:00:00
end_date 2006-06-10 12:00:00
start_date 2006-06-11 12:00:00
end_date 2006-06-12 12:00:00
start_date 2006-06-13 12:00:00
end_date 2006-06-14 12:00:00
start_date 2006-06-15 12:00:00
end_date 2006-06-16 12:00:00
start_date 2006-06-17 12:00:00
end_date 2006-06-18 12:00:00
start_date 2006-06-19 12:00:00
end_date 2006-06-20 12:00:00
start_date 2006-06-21 12:00:00
end_date 2006-06-22 12:00:00
start_date 2006-06-23 12:00:00
end_date 2006-06-24 12:00:00
start_date 2006-06-25 12:00:00
end_date 2006-06-26 12:00:00
start_date 2006-06-27 12:00:00
end_date 2006-06-28 12:00:00
start_date 2006-06-29 12:00:00
end_date 2006-06-30 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                 | 1/15 [03:01<42:14, 181.06s/it]

 13%|███████████▌                                                                           | 2/15 [03:44<21:44, 100.33s/it]

 20%|█████████████████▌                                                                      | 3/15 [04:13<13:32, 67.69s/it]

 27%|███████████████████████▍                                                                | 4/15 [04:38<09:17, 50.65s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [08:29<00:49, 24.56s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [08:49<00:23, 23.15s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [09:13<00:00, 23.33s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [09:13<00:00, 36.91s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/U_2006-06.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [01:26<20:17, 87.00s/it]

 13%|███████████▋                                                                            | 2/15 [03:15<21:33, 99.51s/it]

 20%|█████████████████▌                                                                      | 3/15 [03:35<12:38, 63.19s/it]

 27%|███████████████████████▍                                                                | 4/15 [03:53<08:19, 45.41s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [07:16<00:46, 23.14s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [07:37<00:22, 22.41s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:57<00:00, 21.89s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:57<00:00, 31.85s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/V_2006-06.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                 | 1/15 [01:50<25:42, 110.19s/it]

 13%|███████████▋                                                                            | 2/15 [02:07<12:02, 55.61s/it]

 20%|█████████████████▌                                                                      | 3/15 [02:37<08:45, 43.81s/it]

 27%|███████████████████████▍                                                                | 4/15 [03:14<07:34, 41.34s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [06:36<00:46, 23.44s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [06:55<00:22, 22.18s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:13<00:00, 20.92s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:13<00:00, 28.93s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/W_2006-06.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                 | 1/15 [02:10<30:32, 130.92s/it]

 13%|███████████▋                                                                            | 2/15 [02:31<14:21, 66.25s/it]

 20%|█████████████████▌                                                                      | 3/15 [02:52<09:02, 45.22s/it]

 27%|███████████████████████▍                                                                | 4/15 [03:11<06:24, 34.92s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [07:01<00:43, 21.83s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [07:21<00:21, 21.21s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:39<00:00, 20.36s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:39<00:00, 30.64s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/T_2006-06.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [01:26<20:11, 86.50s/it]

 13%|███████████▋                                                                            | 2/15 [01:45<10:05, 46.59s/it]

 20%|█████████████████▌                                                                      | 3/15 [03:11<12:57, 64.77s/it]

 27%|███████████████████████▍                                                                | 4/15 [03:29<08:30, 46.43s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [10:31<01:04, 32.41s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [10:49<00:28, 28.00s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [11:07<00:00, 24.84s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [11:07<00:00, 44.49s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_2006-06.nc
